In [1]:
from crim_intervals import importScore 
from crim_intervals import main_objs
from community import community_louvain
from copy import deepcopy
from IPython.display import SVG
from ipywidgets import interact
import altair as alt
import glob as glob
import crim_intervals
import crim_intervals.visualizations as viz
import numpy as np
import os
import pandas as pd
import re
import networkx as nx
import requests

MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)

else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)

else:
    print(MUSDIR, "folder already exists.")

saved_csv folder already exists.
Music_Files folder already exists.


In [54]:
# Import MEI File

model = importScore('https://crimproject.org/mei/CRIM_Model_0008.mei')

In [122]:

# These are updated functions.  The originals are on CRIM Intervals here:
# https://github.com/HCDigitalScholarship/intervals/blob/main/crim_intervals/visualizations.py


def tuple_to_single_string(tup):
    return ', '.join(tup)

def generate_hex_colors(n):
    return ['#%02x%02x%02x' % (np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)) for _ in range(n)]

def create_heatmap_dev(x, x2, y, data, heat_map_width, heat_map_height, selector_condition, *selectors, tooltip):
    # convert pattern into string
    # Apply the function to the 'pattern' column
    # data['pattern_str'] = data['pattern'].apply(tuple_to_single_string)
  # Calculate the number of unique values in 'pattern_str'
    num_unique_values = data['pattern_str'].nunique()
    
    # Generate enough hex colors
    hex_colors = generate_hex_colors(num_unique_values)
    
    # Step 2: Assign colors to unique values
    color_map = dict(zip(data['pattern_str'].unique(), hex_colors))
    
    # Add a new column 'color' to the DataFrame
    data['color'] = data['pattern_str'].map(color_map)

    # Create the heatmap with the encoded color
    heatmap = alt.Chart(data).mark_bar().encode(
        x=x,
        x2=x2,
        y=y,
        color='color',
        opacity=alt.condition(selector_condition, alt.value(1), alt.value(0.2)),
        tooltip=tooltip
    ).properties(
        width=heat_map_width,
        height=heat_map_height
    ).add_params(
        *selectors
    )

    return heatmap


def _process_ngrams_df_helper_dev(ngrams_df, main_col):
    """
    The output from the getNgram is usually a table with
    four voices and ngram of notes properties (duration or
    pitch). This method stack this property onto one column
    and mark which voices they are from.
    :param ngrams_df: direct output from getNgram with 1 columns
    for each voices and ngrams of notes' properties.
    :param main_col: the name of the property
    :return: a dataframe with ['start', main_col, 'voice'] as columns
    """
    # copy to avoid changing original ngrams df
    ngrams_df = ngrams_df.copy()

    # add a start column containing offsets
    ngrams_df.index.name = "start"
    ngrams_df = ngrams_df.reset_index().melt(id_vars=["start"], value_name=main_col, var_name="voice")

    ngrams_df["start"] = ngrams_df["start"].astype(float)

    # add new col with pattern as string
    # ngrams_df['pattern_str'] = ngrams_df['pattern'].apply(tuple_to_single_string)

    return ngrams_df


def process_ngrams_df_dev(ngrams_df, ngrams_duration=None, selected_pattern=None, voices=None):
    """
    This method combines ngrams from all voices in different columns
    into one column and calculates the starts and end points of the
    patterns. It could also filter out specific voices or patterns
    for the users to analyze.

    :param ngrams_df: dataframe we got from getNgram in crim-interval
    :param ngrams_duration: if not None, simply output the offsets of the
    ngrams. If we have durations, calculate the end by adding the offsets and
    the durations.
    :param selected_pattern: list of specific patterns the users want (optional)
    :param voices: list of specific voices the users want (optional)
    :return a new, processed dataframe with only desired patterns from desired voices
    combined into one column with start and end points
    """

    ngrams_df = _process_ngrams_df_helper_dev(ngrams_df, 'pattern').dropna()

    ngrams_df['pattern_str'] = ngrams_df['pattern'].apply(tuple_to_single_string)

    if ngrams_duration is not None:
        ngrams_duration = _process_ngrams_df_helper_dev(ngrams_duration, 'duration')
        ngrams_df['end'] = ngrams_df['start'] + ngrams_duration['duration']
    else:
        # make end=start+1 just to display offsets
        ngrams_df['end'] = ngrams_df['start'] + 1

    
    # filter according to voices and patterns (after computing durations for correct offsets)
    if voices:
        voice_condition = ngrams_df['voice'].isin(voices)
        ngrams_df = ngrams_df[voice_condition].dropna(how='all')

    if selected_pattern:
        pattern_condition = ngrams_df['pattern_str'].isin(selected_pattern)
        ngrams_df = ngrams_df[pattern_condition].dropna(how='all')

    return ngrams_df


def _plot_ngrams_df_heatmap_dev(processed_ngrams_df_dev, heatmap_width=800, heatmap_height=300, includeCount=False):
    """
    Plot a heatmap for crim-intervals getNgram's processed output.
    :param ngrams_df: processed crim-intervals getNgram's output.
    :param selected_pattern: list of specific patterns the users want (optional)
    :param voices: list of specific voices the users want (optional)
    :param heatmap_width: the width of the final heatmap (optional)
    :param heatmap_height: the height of the final heatmap (optional)
    :return: a bar chart that displays the different patterns and their counts,
    and a heatmap with the start offsets of chosen voices / patterns
    """

    processed_ngrams_df_dev = processed_ngrams_df_dev.dropna(how='any')
    selector = alt.selection_point(fields=['pattern_str'])
    y = alt.Y("voice", sort=None)

    # make a copy of the processed n_grams and turn them into Strings
    new_processed_ngrams_df_dev = processed_ngrams_df_dev.copy()
    new_processed_ngrams_df_dev['pattern_str'] = processed_ngrams_df_dev['pattern_str'].map(lambda cell: ", ".join(str(item) for item in cell), na_action='ignore')

    heatmap = create_heatmap_dev('start', 'end', y, 'pattern_str', new_processed_ngrams_df_dev, heatmap_width, heatmap_height,
                             selector, selector, tooltip=['start', 'end', 'pattern_str'])
    if includeCount:
        variable = alt.X('pattern_str', axis=alt.Axis(labelAngle=-45))
        patterns_bar = create_bar_chart_dev(variable, 'count(pattern_str)', 'pattern_str', new_processed_ngrams_df_dev, selector, selector)
        return alt.vconcat(patterns_bar, heatmap)
    else:
        return heatmap


def plot_ngrams_heatmap_dev(ngrams_df, ngrams_duration=None, selected_patterns=[], voices=[], heatmap_width=800,
                        heatmap_height=300, includeCount=False):
    """
    Plot a heatmap for crim-intervals getNgram's output.
    :param ngrams_df: crim-intervals getNgram's output
    :param ngrams_duration: if not None, rely on durations in the
    df to calculate the durations of the ngrams.
    :param selected_patterns: list of specific patterns the users want (optional)
    :param voices: list of specific voices the users want (optional)
    :param heatmap_width: the width of the final heatmap (optional)
    :param heatmap_height: the height of the final heatmap (optional)
    :return: a bar chart that displays the different patterns and their counts,
    and a heatmap with the start offsets of chosen voices / patterns
    """
    processed_ngrams_df_dev = process_ngrams_df_dev(ngrams_df, ngrams_duration=ngrams_duration,
                                            selected_pattern=selected_patterns,
                                            voices=voices)
    return _plot_ngrams_df_heatmap_dev(processed_ngrams_df_dev, heatmap_width=heatmap_width, heatmap_height=heatmap_height, includeCount=includeCount)


In [123]:
# _process_ngrams_df_helper_dev(mod_entry_ngrams, 'pattern')

ngrams_df = _process_ngrams_df_helper_dev(mod_entry_ngrams, 'pattern').dropna()
ngrams_df

,start,voice,pattern
0,0.0,[Superius],"(4, 1, 2)"
4,56.0,[Superius],"(-2, -2, -2)"
8,124.0,[Superius],"(1, 1, 2)"
14,222.0,[Superius],"(-2, -2, -2)"
15,244.0,[Superius],"(1, 2, 2)"
...,...,...,...
182,548.0,Bassus,"(2, -3, 2)"
184,640.0,Bassus,"(1, 1, 5)"
193,972.0,Bassus,"(1, -2, -2)"
198,1036.0,Bassus,"(1, -2, -2)"


In [120]:
n=3
combineUnisons=False
kind='d'
thematic = True
anywhere = False

# find entries for model
nr = model.notes(combineUnisons=combineUnisons)
mel = model.melodic(df=nr, kind=kind, compound=False, unit=0, end=False)
mod_mel_ngrams = model.ngrams(df=mel, n=n, exclude=['Rest'])
mod_entry_ngrams = model.entries(df=mel, n=n, thematic=thematic, anywhere=anywhere, exclude=['Rest'])
mod_mel_ngrams_duration = model.durations(df=mel, n=n, mask_df=mod_entry_ngrams)
mod_entries_stack = list(mod_entry_ngrams.stack().unique())


print(model.metadata)
# mod_entries_stack


# This is Existing Call for the 
# display(viz.plot_ngrams_heatmap(mod_entry_ngrams, mod_mel_ngrams_duration, 
#                         selected_patterns=mod_entries_stack,
#                         voices=[],
#                         includeCount=True))

# This is the code to call the DEV versions of the functions above
display(plot_ngrams_heatmap_dev(mod_entry_ngrams, mod_mel_ngrams_duration, 
                        selected_patterns=mod_entries_stack,
                        voices=[],
                        includeCount=True))

# mod_entry_ngrams

{'title': 'Ave Maria', 'composer': 'Josquin Des Prés', 'date': 1502}


TypeError: string indices must be integers

In [121]:
ngrams_df

,start,voice,pattern
0,0.0,[Superius],"(4, 1, 2)"
4,56.0,[Superius],"(-2, -2, -2)"
8,124.0,[Superius],"(1, 1, 2)"
14,222.0,[Superius],"(-2, -2, -2)"
15,244.0,[Superius],"(1, 2, 2)"
...,...,...,...
182,548.0,Bassus,"(2, -3, 2)"
184,640.0,Bassus,"(1, 1, 5)"
193,972.0,Bassus,"(1, -2, -2)"
198,1036.0,Bassus,"(1, -2, -2)"


## Some Tests Below

In [124]:
# output of helper function, which takes in a df of ngrams and then gives us the 'coordinates' of each ngram:
# that is:  the voice part in which it appears, and the start and stop position
# these are used to tell the altair chart where to put the bar

data = process_ngrams_df_dev(mod_entry_ngrams).dropna()
data

,start,voice,pattern,pattern_str,end
0,0.0,[Superius],"(4, 1, 2)","4, 1, 2",1.0
4,56.0,[Superius],"(-2, -2, -2)","-2, -2, -2",57.0
8,124.0,[Superius],"(1, 1, 2)","1, 1, 2",125.0
14,222.0,[Superius],"(-2, -2, -2)","-2, -2, -2",223.0
15,244.0,[Superius],"(1, 2, 2)","1, 2, 2",245.0
...,...,...,...,...,...
182,548.0,Bassus,"(2, -3, 2)","2, -3, 2",549.0
184,640.0,Bassus,"(1, 1, 5)","1, 1, 5",641.0
193,972.0,Bassus,"(1, -2, -2)","1, -2, -2",973.0
198,1036.0,Bassus,"(1, -2, -2)","1, -2, -2",1037.0
